# Problema do 8-Puzzle (Puzzle Deslizante)

O **8-puzzle** é um quebra-cabeça deslizante em um tabuleiro 3×3 com 8 peças numeradas e um espaço vazio. O objetivo é atingir a configuração ordenada deslizando as peças para o espaço vazio.

## Formulação

| Componente | Descrição |
|---|---|
| **Estado** | `PuzzleState(tiles: tuple)` — tupla de 9 inteiros, 0=espaço vazio |
| **Estado inicial** | Configuração fornecida (ex: `(0,8,4,5,3,1,2,7,6)`) |
| **Estado meta** | `(0,1,2,3,4,5,6,7,8)` — peças em ordem |
| **Ações** | `Move(tile)` — deslizar uma peça adjacente ao espaço |
| **Custo** | 1.0 por movimento |

## Representação do Tabuleiro

```
Estado: (0,8,4,5,3,1,2,7,6)    Meta: (0,1,2,3,4,5,6,7,8)
┌───┬───┬───┐                   ┌───┬───┬───┐
│   │ 8 │ 4 │                   │   │ 1 │ 2 │
├───┼───┼───┤                   ├───┼───┼───┤
│ 5 │ 3 │ 1 │                   │ 3 │ 4 │ 5 │
├───┼───┼───┤                   ├───┼───┼───┤
│ 2 │ 7 │ 6 │                   │ 6 │ 7 │ 8 │
└───┴───┴───┘                   └───┴───┴───┘
```

## Complexidade do Espaço de Estados

- Número total de configurações: 9! = 362.880
- Estados alcançáveis (solucionáveis): 9!/2 = **181.440**
- Profundidade ótima de solução pode chegar a 31 movimentos

## Comparação dos Algoritmos

| Algoritmo | Expansões | Custo (passos) | Tempo (ms) |
|---|---|---|---|
| BFS | 177.334 | 28 | 1.608 |
| UCS | 177.334 | 28 | 1.911 |
| IDDFS | 305.076 | 32 | 3.378 |
| A* (peças deslocadas) | 71.630 | 28 | 1.122 |
| A* (Manhattan) | 7.335 | 28 | 89 |
| Hill Climbing | 4 | — | — |

In [1]:
import os, sys
import networkx as nx
import plotly.graph_objects as go
from typing import Any
import math

## Importações do Projeto

Todos os algoritmos de busca são importados para comparação:

- **Busca cega:** `bfs`, `dfs`, `ucs`, `iddfs`, `bidirectional_search`
- **Busca heurística:** `greedy_best_first_search`, `a_star_search`
- **Busca local:** `hill_climbing_search`, `simulated_annealing_search`

O módulo `math` é necessário para calcular `isqrt` (raiz inteira) — o `EightPuzzleProblem` suporta tabuleiros n×n genericamente.

In [ ]:
sys.path.append(os.path.join(os.getcwd(), 'structures'))
from structures.graph import Graph
from structures.problem import SearchResult
from structures.problems.eight_puzzle import EightPuzzleProblem, PuzzleState
from structures.algorithms.blind_search import bfs, dfs, ucs, iddfs, bidirectional_search
from structures.algorithms.heuristic_search import greedy_best_first_search, a_star_search
from structures.algorithms.local_search import hill_climbing_search, simulated_annealing_search

## Criação do Problema

O `EightPuzzleProblem` aceita:
- `initial`: tupla com a configuração inicial (0 = espaço vazio)
- `goal`: configuração meta (padrão: `(0,1,2,3,4,5,6,7,8)`)

Configuração inicial usada: `(0,8,4,5,3,1,2,7,6)` — 8 peças fora do lugar, solução ótima requer **28 movimentos**.

`EightPuzzleProblem.successors` identifica a posição do 0, gera até 4 movimentos possíveis (cima, baixo, esquerda, direita) e troca o 0 com o tile adjacente em cada direção válida.

In [3]:
problem = EightPuzzleProblem((0,8,4,5,3,1,2,7,6))

## BFS — Busca em Largura

O BFS explora o tabuleiro nível por nível (número de movimentos crescente), garantindo a solução com **menor número de movimentos**.

**Resultado:** solução ótima de **28 passos** com 177.334 nós expandidos e 1.608 ms.

O alto custo computacional do BFS se explica pela dimensão do espaço de estados: a solução está na profundidade 28, e o BFS precisa explorar todos os estados até essa profundidade. O conjunto `visited` elimina estados repetidos, limitando as expansões aos ~181.440 estados alcançáveis.

In [6]:
bfs_res = bfs(problem)
bfs_res

SearchResult(found=True, state=PuzzleState(tiles=(0, 1, 2, 3, 4, 5, 6, 7, 8)), actions=[move(5), move(2), move(7), move(3), move(8), move(4), move(1), move(8), move(2), move(7), move(3), move(6), move(8), move(2), move(7), move(3), move(6), move(7), move(4), move(5), move(3), move(4), move(5), move(1), move(2), move(5), move(4), move(3)], path_cost=28.0, expanded=177334, generated=180669, max_frontier=24048, elapsed_ms=1608.0141000002186)

## UCS — Busca de Custo Uniforme

Como todos os movimentos têm custo 1.0 (uniforme), o UCS é **matematicamente equivalente ao BFS**. Ambos expandem os mesmos estados na mesma ordem.

**Resultado:** idêntico ao BFS — 28 passos, 177.334 expansões. O tempo levemente maior (1.911 ms vs. 1.608 ms) se deve ao overhead da fila de prioridade (`heapq`) em comparação com uma fila FIFO simples.

In [7]:
ucs_res = ucs(problem)
ucs_res

SearchResult(found=True, state=PuzzleState(tiles=(0, 1, 2, 3, 4, 5, 6, 7, 8)), actions=[move(5), move(2), move(7), move(3), move(8), move(4), move(1), move(8), move(2), move(7), move(3), move(6), move(8), move(2), move(7), move(3), move(6), move(7), move(4), move(5), move(3), move(4), move(5), move(1), move(2), move(5), move(4), move(3)], path_cost=28.0, expanded=177334, generated=180669, max_frontier=24048, elapsed_ms=1910.7805200001167)

## IDDFS — Busca em Profundidade Iterativa

O IDDFS realiza buscas DFS com profundidade máxima 0, 1, 2, ... até encontrar uma solução. Combina:
- **Eficiência de espaço do DFS:** fronteira máxima de apenas 29 estados
- **Completude do BFS:** encontra solução se existir

**Resultado:** solução de **32 passos** (não ótima!) com 305.076 expansões em 3.378 ms.

A solução não é ótima porque o IDDFS, ao usar DFS internamente, pode encontrar um caminho de comprimento maior que o mínimo. Os estados re-expandidos em cada iteração explicam o alto número de expansões — o `SearchResult` acumula as métricas de *todas* as iterações DFS.

In [9]:
iddfs_res = iddfs(problem, max_depth=9999)
iddfs_res

SearchResult(found=True, state=PuzzleState(tiles=(0, 1, 2, 3, 4, 5, 6, 7, 8)), actions=[move(5), move(2), move(7), move(3), move(8), move(4), move(1), move(6), move(3), move(8), move(6), move(1), move(4), move(5), move(2), move(6), move(1), move(3), move(8), move(7), move(6), move(1), move(3), move(4), move(5), move(2), move(1), move(3), move(4), move(5), move(2), move(1)], path_cost=32.0, expanded=305076, generated=486089, max_frontier=29, elapsed_ms=3378.200979999747)

## DFS — Busca em Profundidade (sem limite)

O DFS sem limite de profundidade pode produzir soluções extremamente longas no 8-puzzle. O output não é exibido completamente devido ao tamanho.

**Características do DFS no 8-puzzle:**
- Pode encontrar soluções de centenas ou milhares de movimentos
- Uso de memória muito baixo (fronteira linear na profundidade)
- **Não prático** para encontrar soluções curtas no 8-puzzle

O DFS é inadequado para este problema por dois motivos: (1) não garante solução ótima e (2) pode se perder em ramos muito profundos antes de explorar soluções mais curtas.

In [4]:
dfs_res = dfs(problem)
dfs_res

SearchResult(found=True, state=PuzzleState(tiles=(0, 1, 2, 3, 4, 5, 6, 7, 8)), actions=[move(8), move(4), move(1), move(3), move(5), move(2), move(7), move(6), move(3), move(5), move(2), move(7), move(6), move(3), move(5), move(2), move(7), move(6), move(3), move(5), move(2), move(7), move(6), move(3), move(5), move(2), move(7), move(6), move(3), move(5), move(2), move(3), move(6), move(7), move(3), move(2), move(5), move(6), move(7), move(3), move(2), move(5), move(6), move(7), move(3), move(1), move(4), move(8), move(7), move(3), move(1), move(2), move(5), move(6), move(3), move(1), move(2), move(5), move(6), move(3), move(1), move(2), move(5), move(6), move(3), move(1), move(2), move(5), move(6), move(3), move(1), move(2), move(5), move(6), move(3), move(1), move(2), move(3), move(1), move(2), move(3), move(5), move(6), move(1), move(2), move(3), move(5), move(6), move(1), move(2), move(3), move(4), move(8), move(7), move(2), move(3), move(4), move(5), move(6), move(1), move(3), mov

## A* com Heurística de Peças Deslocadas

### Heurística: Peças Fora do Lugar (*Misplaced Tiles*)

`heuristic_misplaced_tiles(state)` conta o número de peças **que não estão na posição meta**, excluindo o espaço vazio.

**Propriedades:**
- **Admissível:** cada peça fora do lugar precisa de pelo menos 1 movimento para chegar ao lugar correto
- **Consistente:** mover uma peça reduz h no máximo em 1, e o custo é 1 → satisfaz h(n) ≤ 1 + h(n')

**Resultado:** solução ótima de **28 passos** com 71.630 expansões (2,5× menos que BFS) em 1.122 ms.

A heurística admissível garante que o A* encontre a solução ótima. A redução de expansões demonstra o poder da busca informada.

In [12]:
def heuristic_misplaced_tiles(state: PuzzleState) -> int:
    return sum(1 for i, tile in enumerate(state.tiles) if tile != 0 and tile != i)

a_star_res = a_star_search(problem, heuristic_misplaced_tiles)
a_star_res

SearchResult(found=True, state=PuzzleState(tiles=(0, 1, 2, 3, 4, 5, 6, 7, 8)), actions=[move(5), move(3), move(8), move(4), move(1), move(8), move(7), move(6), move(8), move(1), move(4), move(5), move(3), move(2), move(6), move(7), move(1), move(4), move(5), move(1), move(2), move(3), move(1), move(2), move(4), move(5), move(2), move(1)], path_cost=28.0, expanded=71630, generated=93907, max_frontier=22279, elapsed_ms=1122.4852099985583)

## Heurística da Distância de Manhattan

`heuristic_manhattan_distance(state)` calcula a soma das **distâncias de Manhattan** de cada peça até sua posição meta: para cada peça, a distância é |linha_atual - linha_meta| + |coluna_atual - coluna_meta|.

**Por que Manhattan domina peças deslocadas?**
- Para toda configuração: h_manhattan(n) ≥ h_misplaced(n) — é mais informativa
- Uma peça deslocada contribui com pelo menos 1 para Manhattan (pode ser mais)
- Mais próxima de h* (custo real), portanto expande menos nós

**Propriedades:**
- **Admissível:** cada peça precisa de pelo menos sua distância de Manhattan em movimentos (ignorando interferências com outras peças)
- **Consistente:** mover qualquer peça muda sua distância de Manhattan em ±1

In [15]:
def heuristic_manhattan_distance(state: PuzzleState) -> int:
    n = int(math.sqrt(len(state.tiles)))
    if n * n != len(state.tiles):
        raise ValueError("The number of tiles must be a perfect square.")
    
    goal_pos = {tile: divmod(i, n) for i, tile in enumerate(range(n * n))}
    distance = 0
    for i, tile in enumerate(state.tiles):
        if tile == 0:
            continue
        r, c = divmod(i, n)
        gr, gc = goal_pos[tile]
        distance += abs(gr - r) + abs(gc - c)
    return distance

## A* com Distância de Manhattan

**Resultado:** solução ótima de **28 passos** com apenas 7.335 expansões em 89 ms.

**Comparação de heurísticas:**

| Heurística | Expansões | Tempo (ms) | Nós gerados |
|---|---|---|---|
| Nenhuma (BFS/UCS) | 177.334 | ~1.800 | ~181.000 |
| Peças deslocadas | 71.630 | 1.122 | 93.907 |
| Manhattan | 7.335 | 89 | 11.134 |

A distância de Manhattan é **24× mais eficiente** que BFS e **10× mais eficiente** que peças deslocadas. Isso demonstra que heurísticas mais informativas reduzem dramaticamente o espaço de busca, mantendo a otimalidade da solução.

In [16]:
a_star_res_manhattan = a_star_search(problem, heuristic_manhattan_distance)
a_star_res_manhattan

SearchResult(found=True, state=PuzzleState(tiles=(0, 1, 2, 3, 4, 5, 6, 7, 8)), actions=[move(5), move(3), move(8), move(4), move(1), move(8), move(7), move(6), move(8), move(1), move(4), move(5), move(3), move(2), move(6), move(7), move(1), move(4), move(5), move(1), move(2), move(3), move(1), move(2), move(4), move(5), move(2), move(1)], path_cost=28.0, expanded=7335, generated=11134, max_frontier=3800, elapsed_ms=88.74975999970047)

## Hill Climbing

O *hill climbing* é um algoritmo de **busca local**: a partir do estado atual, move-se para o vizinho com **menor valor de h(n)**. Não mantém um caminho — apenas o estado corrente.

**Resultado:** `found=False` — o algoritmo ficou preso em um **ótimo local** após apenas 4 expansões.

No 8-puzzle, o hill climbing falha com frequência porque:
1. **Ótimos locais:** configurações onde todos os vizinhos têm h maior que o estado atual
2. **Platôs:** regiões onde h é igual em todos os vizinhos
3. **Sem backtracking:** uma vez preso, não há como escapar

O hill climbing é útil para problemas com paisagem de energia bem comportada, mas o 8-puzzle tem muitos ótimos locais que o tornam inadequado.

In [26]:
hill_climbing_res = hill_climbing_search(problem, heuristic_manhattan_distance)
hill_climbing_res

SearchResult(found=False, state=None, actions=[], path_cost=inf, expanded=4, generated=4, max_frontier=1, elapsed_ms=4.493720000027679)

## 15-Puzzle — Problema Mais Difícil

O **15-puzzle** usa um tabuleiro 4×4 com 15 peças e 1 espaço vazio. É exponencialmente mais difícil que o 8-puzzle:

- Espaço de estados: 16!/2 ≈ 10^13 estados alcançáveis
- Profundidade ótima média: ~52 movimentos
- BFS/UCS são **completamente impraticáveis** (memória insuficiente)
- A* com Manhattan ainda funciona, mas é muito mais lento

Para este problema extremo, utiliza-se **Simulated Annealing** — um algoritmo de busca local estocástico que pode escapar de ótimos locais.

Configuração inicial: `(0,8,9,13,4,11,12,5,3,1,15,2,7,14,6,10)` (tabuleiro 4×4)

In [23]:
problem = EightPuzzleProblem((0,8,9,13,4,11,12,5,3,1,15,2,7,14,6,10), goal=(0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15))

## Simulated Annealing

O *Simulated Annealing* é inspirado no processo de **recozimento (annealing)** da metalurgia: aquecer um material e resfriá-lo lentamente para minimizar defeitos na estrutura cristalina.

### Algoritmo:
1. A cada passo, gera um **vizinho aleatório** do estado atual
2. Se o vizinho é melhor (h menor): aceita sempre
3. Se é pior (h maior): aceita com probabilidade `e^(-Δh/T)` — temperatura alta = mais aceitação de pioras
4. A temperatura **T** decresce ao longo do tempo (cooling schedule)

### Parâmetros utilizados:

| Parâmetro | Valor | Significado |
|---|---|---|
| `initial_temperature` | 50.0 | Temperatura inicial (aceita muitas pioras) |
| `cooling_rate` | 0.995 | Multiplicador por iteração (resfriamento lento) |
| `steps_per_temp` | 100 | Movimentos antes de baixar temperatura |
| `max_steps` | 300.000 | Limite máximo de iterações |

**Resultado esperado:** o SA consegue escapar de ótimos locais e eventualmente encontrar a solução do 15-puzzle. A qualidade da solução depende dos hiperparâmetros e da semente aleatória.

In [27]:
sa_res = simulated_annealing_search(
    problem,
    heuristic=heuristic_manhattan_distance,
    initial_temperature=50.0,
    cooling_rate=0.995,
    steps_per_temp=100,
    max_steps=300_000,
    seed=0,
)
sa_res

SearchResult(found=True, state=PuzzleState(tiles=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15)), actions=[move(8), move(8), move(8), move(8), move(8), move(8), move(4), move(3), move(3), move(11), move(1), move(3), move(11), move(1), move(1), move(11), move(7), move(14), move(6), move(10), move(10), move(10), move(10), move(10), move(2), move(2), move(2), move(2), move(2), move(15), move(12), move(12), move(10), move(10), move(10), move(2), move(2), move(6), move(14), move(7), move(7), move(7), move(3), move(3), move(7), move(7), move(7), move(14), move(14), move(7), move(11), move(4), move(8), move(8), move(4), move(4), move(4), move(1), move(1), move(1), move(3), move(10), move(15), move(15), move(10), move(3), move(8), move(4), move(4), move(8), move(1), move(4), move(4), move(4), move(8), move(8), move(4), move(1), move(8), move(8), move(3), move(3), move(1), move(4), move(4), move(4), move(4), move(11), move(11), move(1), move(12), move(9), move(8), move(12), move(1), mov